# E4 — Distribution-Shift Audit
## Leave-One-Domain-Out (LODO) Stressor-Domain Shift — Calib-BERT v2

E4 tests **true domain shift inside Dreaddit** using the dataset's original five-domain taxonomy.

### Why this is a genuine shift experiment
For each held-out target domain:

1. **all examples from that domain are removed from classifier training**;
2. **all examples from that domain are removed from calibration fitting**;
3. Vanilla BERT and LS ε=0.10 are trained only on the other four source domains;
4. Temperature Scaling is fitted only on the source-domain calibration subset;
5. the held-out domain is evaluated only on the official Dreaddit test partition.

Thus the target domain is unseen by both the neural classifier and the post-hoc calibrator.

### Five original Dreaddit domains
- abuse
- anxiety
- financial
- PTSD
- social

### Predeclared model conditions
- Vanilla BERT
- Vanilla BERT + Temperature Scaling
- BERT + Label Smoothing ε=0.10

### Architecture choice
E4 uses **BERT-base-uncased as the fixed anchor architecture**, because BERT is the original Calib-BERT architecture. It is **not chosen because of E3 official-test performance**. Cross-backbone generalization has already been evaluated separately in E3.

### E4 outputs
E4 produces:
- per-domain target-OOD metrics,
- source-domain reference metrics,
- target-minus-source shift deltas,
- macro-domain summaries,
- pooled LODO OOD predictions over all 715 official-test instances,
- calibration/selective-reliability figures,
- a prediction-level file ready for E7 cluster-aware inference.

### Important terminology
This notebook calls the held-out-domain evaluation **OOD/domain-shift**.  
A simple subgroup analysis of a model trained on all five domains would *not* be called OOD.

Temporal metadata are audited, but temporal shift is **not** used as a primary E4 experiment because time and domain composition are partially confounded in Dreaddit.


In [ ]:
# ============================================================
# 0. USER CONFIGURATION
# ============================================================
from pathlib import Path

# Recommended: keep True so completed folds survive Colab runtime loss.
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive/CalibBERT_E4")
    except Exception as e:
        print("Google Drive mount failed:", repr(e))
        print("Falling back to /content. Completed folds will NOT survive runtime loss.")
        PROJECT_ROOT = Path("/content/calibbert_v2_e4")
else:
    PROJECT_ROOT = Path("/content/calibbert_v2_e4")

# Upload these three files to /content:
# 1) dreaddit-train.csv or dreaddit-train(1).csv
# 2) dreaddit-test.csv or dreaddit-test(1).csv
# 3) calibbert_e2_lite.zip
INPUT_DIR = Path("/content")

WORK_DIR = PROJECT_ROOT / "work"
CACHE_DIR = PROJECT_ROOT / "hf_cache"
OUT_DIR = PROJECT_ROOT / "outputs_e4"

for p in [WORK_DIR, CACHE_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PRIMARY_SEED = 42
MODEL_ID = "bert-base-uncased"

MAX_LENGTH = 128
EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 0
GRAD_CLIP = 1.0
LS_EPSILON = 0.10

EXPECTED_TRAIN_N = 2838
EXPECTED_TEST_N = 715

EXPECTED_TRAIN_SHA256 = "4e791e60d3f04dd9f14abb1228b0d4f92bf628cc84097c1c173401e98d355f4e"
EXPECTED_TEST_SHA256 = "a3b6c9bf5d685f49d57e05d24266dddc52999843256b13e65c07267c543175fe"

DOMAINS = ["abuse", "anxiety", "financial", "PTSD", "social"]

DOMAIN_MAP = {
    "domesticviolence": "abuse",
    "survivorsofabuse": "abuse",
    "anxiety": "anxiety",
    "stress": "anxiety",
    "almosthomeless": "financial",
    "assistance": "financial",
    "food_pantry": "financial",
    "homeless": "financial",
    "ptsd": "PTSD",
    "relationships": "social",
}

FIXED_COVERAGES = [0.90, 0.80, 0.70, 0.50]

print("Project root:", PROJECT_ROOT)
print("Outputs     :", OUT_DIR)


## 1. Environment preparation

Use a **T4 GPU**. E4 performs 10 BERT fine-tuning runs:

- 5 held-out domains × Vanilla
- 5 held-out domains × LS 0.10

Temperature Scaling does not require neural-network retraining.

Because each completed run is cached immediately, rerunning the notebook resumes from existing fold outputs.


In [ ]:
import os
import sys
import gc
import json
import math
import random
import hashlib
import zipfile
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

import sklearn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    log_loss,
)

import scipy
from scipy.optimize import minimize_scalar

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

environment = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "transformers": transformers.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print(json.dumps(environment, indent=2))
(OUT_DIR / "e4_environment.json").write_text(json.dumps(environment, indent=2))

if not torch.cuda.is_available():
    raise RuntimeError(
        "E4 neural-network training requires a GPU runtime. "
        "In Colab choose Runtime > Change runtime type > T4 GPU."
    )


## 2. Auto-discover and verify canonical inputs

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_by_hash(directory, expected_hash, suffix=".csv"):
    for p in Path(directory).glob(f"*{suffix}"):
        try:
            if sha256_file(p) == expected_hash:
                return p
        except Exception:
            pass
    return None

TRAIN_PATH = find_by_hash(INPUT_DIR, EXPECTED_TRAIN_SHA256)
TEST_PATH = find_by_hash(INPUT_DIR, EXPECTED_TEST_SHA256)
LITE_ZIP = INPUT_DIR / "calibbert_e2_lite.zip"

if TRAIN_PATH is None:
    raise FileNotFoundError("Canonical Dreaddit train CSV not found in /content.")
if TEST_PATH is None:
    raise FileNotFoundError("Canonical Dreaddit test CSV not found in /content.")
if not LITE_ZIP.exists():
    raise FileNotFoundError("calibbert_e2_lite.zip not found in /content.")

assert sha256_file(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
assert sha256_file(TEST_PATH) == EXPECTED_TEST_SHA256

print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)
print("Lite :", LITE_ZIP)
print("\nCanonical raw-data hashes: PASS")


## 3. Load raw metadata and original Dreaddit domain taxonomy

The source CSVs contain the metadata needed for E4, including:
- `subreddit`
- `post_id`
- `social_timestamp`
- `label`

The domain map is fixed from the original Dreaddit taxonomy and is **not learned from the official test labels**.


In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

assert len(train_raw) == EXPECTED_TRAIN_N
assert len(test_raw) == EXPECTED_TEST_N

required_raw_cols = {
    "subreddit", "post_id", "text", "id", "label", "social_timestamp"
}
for name, df in [("train", train_raw), ("test", test_raw)]:
    missing = required_raw_cols - set(df.columns)
    if missing:
        raise KeyError(f"{name} missing columns: {sorted(missing)}")

for df in [train_raw, test_raw]:
    df["domain"] = df["subreddit"].map(DOMAIN_MAP)
    df["datetime_utc"] = pd.to_datetime(
        df["social_timestamp"], unit="s", utc=True, errors="coerce"
    )
    df["year"] = df["datetime_utc"].dt.year

if train_raw["domain"].isna().any() or test_raw["domain"].isna().any():
    bad = sorted(
        set(train_raw.loc[train_raw["domain"].isna(), "subreddit"].astype(str))
        | set(test_raw.loc[test_raw["domain"].isna(), "subreddit"].astype(str))
    )
    raise RuntimeError(f"Unmapped subreddits found: {bad}")

assert set(train_raw["domain"].unique()) == set(DOMAINS)
assert set(test_raw["domain"].unique()) == set(DOMAINS)

print("Domain taxonomy mapping: PASS")
display(
    pd.crosstab(
        [train_raw["domain"]],
        train_raw["label"],
        margins=True
    )
)
display(
    pd.crosstab(
        [test_raw["domain"]],
        test_raw["label"],
        margins=True
    )
)


## 4. Metadata audit: domain balance and temporal confounding

This cell documents why E4 uses **domain LODO** as the primary shift experiment.

Dreaddit spans 2017–2018, but calendar time is not orthogonal to domain.  
For example, the social/relationships domain occupies a narrow temporal window in this release. A naïve early-vs-late split would therefore mix temporal and domain shift.

The notebook records this instead of making an unsupported claim of isolated temporal robustness.


In [ ]:
def domain_metadata_table(df, split_name):
    return (
        df.groupby("domain")
        .agg(
            n=("label", "size"),
            unique_posts=("post_id", "nunique"),
            stress_rate=("label", "mean"),
            date_min=("datetime_utc", "min"),
            date_max=("datetime_utc", "max"),
        )
        .reset_index()
        .assign(dataset_split=split_name)
    )

metadata_audit = pd.concat([
    domain_metadata_table(train_raw, "development"),
    domain_metadata_table(test_raw, "official_test"),
], ignore_index=True)

metadata_audit.to_csv(
    OUT_DIR / "e4_domain_metadata_audit.csv",
    index=False
)

year_domain_train = pd.crosstab(train_raw["domain"], train_raw["year"])
year_domain_test = pd.crosstab(test_raw["domain"], test_raw["year"])

year_domain_train.to_csv(
    OUT_DIR / "e4_domain_year_crosstab_development.csv"
)
year_domain_test.to_csv(
    OUT_DIR / "e4_domain_year_crosstab_test.csv"
)

display(metadata_audit)
print("\nDevelopment domain × year:")
display(year_domain_train)
print("\nOfficial test domain × year:")
display(year_domain_test)


## 5. Extract and verify the archived post-grouped split

In [ ]:
LITE_DIR = WORK_DIR / "e2_lite"
LITE_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LITE_ZIP, "r") as z:
    z.extractall(LITE_DIR)

SPLIT_PATH = LITE_DIR / "development_split_assignment.csv"
if not SPLIT_PATH.exists():
    raise FileNotFoundError("development_split_assignment.csv missing from lite ZIP.")

split = pd.read_csv(SPLIT_PATH)

check = train_raw.loc[
    split["row_index"].astype(int),
    ["id", "post_id", "label", "domain"]
].reset_index(drop=True)

assert np.array_equal(
    check["id"].astype(str).to_numpy(),
    split["id"].astype(str).to_numpy()
)
assert np.array_equal(
    check["post_id"].astype(str).to_numpy(),
    split["post_id"].astype(str).to_numpy()
)
assert np.array_equal(
    check["label"].astype(int).to_numpy(),
    split["label"].astype(int).to_numpy()
)

split = split.copy()
split["domain"] = check["domain"].to_numpy()

split_counts = pd.crosstab(split["domain"], split["split"])
split_counts.to_csv(OUT_DIR / "e4_domain_by_archived_split.csv")

print("Archived split alignment: PASS")
display(split_counts)


## 6. Predeclare LODO folds and write the E4 lock

For a held-out target domain `d`:

- **classifier training** = archived fit + validation rows from the other four domains;
- **calibration fitting** = archived calibration rows from the other four domains;
- **target OOD test** = official-test rows from `d`;
- **source-domain reference test** = official-test rows from the other four domains.

No target-domain example is used for neural training or Temperature Scaling.


In [ ]:
fold_rows = []

for heldout in DOMAINS:
    train_source_idx = split.loc[
        split["split"].isin(["fit", "validation"])
        & (split["domain"] != heldout),
        "row_index"
    ].astype(int).to_numpy()

    cal_source_idx = split.loc[
        (split["split"] == "calibration")
        & (split["domain"] != heldout),
        "row_index"
    ].astype(int).to_numpy()

    target_test_idx = test_raw.index[
        test_raw["domain"] == heldout
    ].to_numpy()

    source_test_idx = test_raw.index[
        test_raw["domain"] != heldout
    ].to_numpy()

    fold_rows.append({
        "heldout_domain": heldout,
        "source_train_n": len(train_source_idx),
        "source_calibration_n": len(cal_source_idx),
        "target_test_n": len(target_test_idx),
        "source_reference_test_n": len(source_test_idx),
        "source_train_stress_rate": float(
            train_raw.iloc[train_source_idx]["label"].mean()
        ),
        "source_calibration_stress_rate": float(
            train_raw.iloc[cal_source_idx]["label"].mean()
        ),
        "target_test_stress_rate": float(
            test_raw.iloc[target_test_idx]["label"].mean()
        ),
        "source_reference_test_stress_rate": float(
            test_raw.iloc[source_test_idx]["label"].mean()
        ),
    })

fold_manifest = pd.DataFrame(fold_rows)
fold_manifest.to_csv(
    OUT_DIR / "e4_lodo_fold_manifest.csv",
    index=False
)

display(fold_manifest)

# Resolve immutable BERT model revision before any E4 training.
cfg = AutoConfig.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
MODEL_REVISION = getattr(cfg, "_commit_hash", None)

if not MODEL_REVISION:
    raise RuntimeError("Could not resolve immutable BERT model revision.")

model_revision_lock = {
    "model_id": MODEL_ID,
    "revision_commit": MODEL_REVISION,
    "resolved_utc": datetime.now(timezone.utc).isoformat(),
}

(OUT_DIR / "E4_MODEL_REVISION_LOCK.json").write_text(
    json.dumps(model_revision_lock, indent=2)
)

e4_lock = {
    "analysis": "E4 Leave-One-Domain-Out Distribution-Shift Audit",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_train_sha256": EXPECTED_TRAIN_SHA256,
    "canonical_test_sha256": EXPECTED_TEST_SHA256,
    "domain_taxonomy": DOMAIN_MAP,
    "domains": DOMAINS,
    "model_anchor": {
        "architecture": "BERT-base-uncased",
        "selection_reason": (
            "original Calib-BERT anchor architecture; "
            "not selected from E3 official-test performance"
        ),
        "revision_commit": MODEL_REVISION,
    },
    "conditions": [
        "Vanilla",
        "Vanilla+TemperatureScaling",
        "LS0.10",
    ],
    "training": {
        "seed": PRIMARY_SEED,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": TRAIN_BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "max_length": MAX_LENGTH,
        "warmup_steps": WARMUP_STEPS,
        "gradient_clip": GRAD_CLIP,
        "label_smoothing_epsilon": LS_EPSILON,
    },
    "classifier_training_rule": (
        "archived fit+validation rows from four source domains only"
    ),
    "temperature_fit_rule": (
        "archived calibration rows from four source domains only"
    ),
    "target_evaluation_rule": (
        "official-test examples from the held-out fifth domain only"
    ),
    "source_reference_rule": (
        "official-test examples from the same four source domains"
    ),
    "test_based_method_selection": False,
    "target_domain_used_for_training": False,
    "target_domain_used_for_calibrator_fit": False,
    "temporal_shift_primary": False,
    "temporal_shift_note": (
        "Temporal metadata are audited but not used as an isolated primary shift "
        "because domain and calendar time are partially confounded in Dreaddit."
    ),
    "formal_cluster_inference_deferred_to": "E7",
}

(OUT_DIR / "E4_ANALYSIS_LOCK.json").write_text(
    json.dumps(e4_lock, indent=2)
)

print("E4 design lock written BEFORE training.")
print("BERT revision:", MODEL_REVISION)


## 7. Deterministic training helpers

In [ ]:
def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

class TextDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length):
        texts = frame["text"].fillna("").astype(str).tolist()
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(
            frame["label"].astype(int).to_numpy(),
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item["labels"] = self.labels[idx]
        return item

def make_loader(ds, batch_size, shuffle, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=g,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

def soft_target_cross_entropy(logits, labels, epsilon):
    K = logits.shape[-1]
    target = torch.full(
        (labels.shape[0], K),
        epsilon / (K - 1),
        dtype=torch.float32,
        device=labels.device,
    )
    target.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)
    return -(target * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

@torch.no_grad()
def predict_logits(model, dataset):
    loader = make_loader(
        dataset, EVAL_BATCH_SIZE, False, PRIMARY_SEED
    )
    model.eval()
    logits_all, labels_all = [], []

    for batch in loader:
        labels = batch.pop("labels")
        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
        }
        logits = model(**inputs).logits

        if not torch.isfinite(logits).all():
            raise RuntimeError("Non-finite logits detected during evaluation.")

        logits_all.append(logits.detach().cpu().numpy())
        labels_all.append(labels.numpy())

    return np.concatenate(logits_all), np.concatenate(labels_all)


## 8. Train/reuse one LODO model condition

Each completed fold-condition is cached immediately.  
If the runtime disconnects, re-upload the three input files, rerun the notebook from the top, and completed runs are reused from Google Drive.


In [ ]:
def build_fold_frames(heldout):
    source_train_idx = split.loc[
        split["split"].isin(["fit", "validation"])
        & (split["domain"] != heldout),
        "row_index"
    ].astype(int).to_numpy()

    source_cal_idx = split.loc[
        (split["split"] == "calibration")
        & (split["domain"] != heldout),
        "row_index"
    ].astype(int).to_numpy()

    target_test_idx = test_raw.index[
        test_raw["domain"] == heldout
    ].to_numpy()

    source_test_idx = test_raw.index[
        test_raw["domain"] != heldout
    ].to_numpy()

    return (
        train_raw.iloc[source_train_idx].reset_index(drop=True),
        train_raw.iloc[source_cal_idx].reset_index(drop=True),
        test_raw.iloc[target_test_idx].reset_index(drop=True),
        test_raw.iloc[source_test_idx].reset_index(drop=True),
    )

def train_lodo_condition(heldout, mode):
    condition = "Vanilla" if mode == "none" else "LS0.10"

    cache_npz = (
        OUT_DIR /
        f"E4_LODO_{heldout}_{condition}_seed{PRIMARY_SEED}_logits.npz"
    )
    history_csv = (
        OUT_DIR /
        f"E4_LODO_{heldout}_{condition}_seed{PRIMARY_SEED}_history.csv"
    )

    if cache_npz.exists():
        print(f"[REUSE] {cache_npz.name}")
        arr = np.load(cache_npz)

        for key in [
            "cal_logits", "target_logits", "source_logits"
        ]:
            if not np.isfinite(arr[key]).all():
                raise RuntimeError(
                    f"Cached file contains non-finite values: {cache_npz}"
                )

        return {
            "cal_logits": arr["cal_logits"],
            "cal_y": arr["cal_y"],
            "target_logits": arr["target_logits"],
            "target_y": arr["target_y"],
            "source_logits": arr["source_logits"],
            "source_y": arr["source_y"],
            "history": (
                pd.read_csv(history_csv)
                if history_csv.exists()
                else pd.DataFrame()
            ),
        }

    source_train_df, source_cal_df, target_test_df, source_test_df = (
        build_fold_frames(heldout)
    )

    print("\n====================================================")
    print(f"LODO target={heldout} / condition={condition}")
    print("====================================================")
    print("source train:", len(source_train_df))
    print("source cal  :", len(source_cal_df))
    print("target test :", len(target_test_df))
    print("source test :", len(source_test_df))

    set_all_seeds(PRIMARY_SEED)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=CACHE_DIR,
        use_fast=True,
    )

    train_ds = TextDataset(
        source_train_df, tokenizer, MAX_LENGTH
    )
    cal_ds = TextDataset(
        source_cal_df, tokenizer, MAX_LENGTH
    )
    target_ds = TextDataset(
        target_test_df, tokenizer, MAX_LENGTH
    )
    source_test_ds = TextDataset(
        source_test_df, tokenizer, MAX_LENGTH
    )

    train_loader = make_loader(
        train_ds,
        TRAIN_BATCH_SIZE,
        True,
        PRIMARY_SEED
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=CACHE_DIR,
        num_labels=2,
        torch_dtype=torch.float32,
    ).float().to(DEVICE)

    # Pre-training finite gate
    first_batch = next(iter(train_loader))
    first_inputs = {
        k: v.to(DEVICE)
        for k, v in first_batch.items()
        if k != "labels"
    }

    model.eval()
    with torch.no_grad():
        initial_logits = model(**first_inputs).logits

    if not torch.isfinite(initial_logits).all():
        raise RuntimeError(
            f"{heldout}/{condition}: non-finite logits before training."
        )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps,
    )

    history_rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = []

        for step, batch in enumerate(train_loader, start=1):
            optimizer.zero_grad(set_to_none=True)

            labels = batch["labels"].to(DEVICE)
            inputs = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
                if k != "labels"
            }

            logits = model(**inputs).logits

            if not torch.isfinite(logits).all():
                raise RuntimeError(
                    f"{heldout}/{condition}: non-finite logits "
                    f"epoch={epoch}, step={step}"
                )

            if mode == "none":
                loss = F.cross_entropy(logits, labels)
            elif mode == "ls":
                loss = soft_target_cross_entropy(
                    logits, labels, LS_EPSILON
                )
            else:
                raise ValueError(mode)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"{heldout}/{condition}: non-finite loss "
                    f"epoch={epoch}, step={step}"
                )

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), GRAD_CLIP
            )

            if not torch.isfinite(grad_norm):
                raise RuntimeError(
                    f"{heldout}/{condition}: non-finite gradient norm "
                    f"epoch={epoch}, step={step}"
                )

            optimizer.step()
            scheduler.step()

            running.append(float(loss.detach().cpu()))

        mean_loss = float(np.mean(running))
        history_rows.append({
            "heldout_domain": heldout,
            "condition": condition,
            "seed": PRIMARY_SEED,
            "epoch": epoch,
            "train_loss": mean_loss,
        })

        print(
            f"{heldout} / {condition} / epoch {epoch}/{EPOCHS}: "
            f"loss={mean_loss:.6f}"
        )

    cal_logits, cal_y = predict_logits(model, cal_ds)
    target_logits, target_y = predict_logits(model, target_ds)
    source_logits, source_y = predict_logits(model, source_test_ds)

    for name, arr in [
        ("calibration", cal_logits),
        ("target", target_logits),
        ("source", source_logits),
    ]:
        if not np.isfinite(arr).all():
            raise RuntimeError(
                f"{heldout}/{condition}: non-finite {name} logits."
            )

    np.savez_compressed(
        cache_npz,
        cal_logits=cal_logits,
        cal_y=cal_y,
        target_logits=target_logits,
        target_y=target_y,
        source_logits=source_logits,
        source_y=source_y,
    )

    history = pd.DataFrame(history_rows)
    history.to_csv(history_csv, index=False)

    del model, tokenizer
    del train_ds, cal_ds, target_ds, source_test_ds, train_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "cal_logits": cal_logits,
        "cal_y": cal_y,
        "target_logits": target_logits,
        "target_y": target_y,
        "source_logits": source_logits,
        "source_y": source_y,
        "history": history,
    }


## 9. Run all five LODO folds

This is the computationally expensive E4 cell.

It performs at most **10 BERT training runs**.  
Completed runs are reused automatically.

If the runtime stops after some folds, reconnect and rerun from the top; cached folds in Google Drive are not retrained.


In [ ]:
runs = {}
history_parts = []

for heldout in DOMAINS:
    for mode in ["none", "ls"]:
        key = (heldout, mode)
        result = train_lodo_condition(heldout, mode)
        runs[key] = result

        if len(result["history"]):
            history_parts.append(result["history"])

if history_parts:
    training_history = pd.concat(
        history_parts, ignore_index=True
    )
    training_history.to_csv(
        OUT_DIR / "e4_training_history.csv",
        index=False
    )

print("\nAll E4 LODO neural runs complete.")


## 10. Temperature Scaling on source-domain calibration only

In [ ]:
def softmax_np(logits):
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def nll_from_logits(logits, y):
    return float(
        log_loss(
            y,
            softmax_np(logits),
            labels=[0, 1]
        )
    )

def fit_temperature(cal_logits, y):
    def objective(logT):
        T = float(np.exp(logT))
        return nll_from_logits(cal_logits / T, y)

    result = minimize_scalar(
        objective,
        bounds=(-4.0, 4.0),
        method="bounded",
        options={"xatol": 1e-10},
    )

    T = float(np.exp(result.x))

    return {
        "temperature": T,
        "calibration_nll_before": nll_from_logits(cal_logits, y),
        "calibration_nll_after": nll_from_logits(cal_logits / T, y),
        "optimizer_success": bool(result.success),
    }

temperatures = {}
temperature_rows = []

for heldout in DOMAINS:
    r = runs[(heldout, "none")]
    fit = fit_temperature(r["cal_logits"], r["cal_y"])
    temperatures[heldout] = fit["temperature"]

    temperature_rows.append({
        "heldout_domain": heldout,
        "source_calibration_n": len(r["cal_y"]),
        **fit,
    })

temperature_table = pd.DataFrame(temperature_rows)
temperature_table.to_csv(
    OUT_DIR / "e4_temperature_scaling_parameters.csv",
    index=False
)

display(temperature_table)


## 11. Metrics

In [ ]:
def ece_equal_width(y_true, probs, n_bins=10):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        mask = (conf >= lo) & (
            (conf <= hi)
            if b == n_bins - 1
            else (conf < hi)
        )

        if mask.any():
            ece += mask.mean() * abs(
                correct[mask].mean()
                - conf[mask].mean()
            )

    return float(ece)

def ace_equal_frequency(y_true, probs, n_bins=10):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)
    order = np.argsort(conf)

    ace = 0.0
    for idx in np.array_split(order, n_bins):
        if len(idx):
            ace += (len(idx) / len(y_true)) * abs(
                correct[idx].mean()
                - conf[idx].mean()
            )
    return float(ace)

def selective_metrics(y_true, probs):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)

    # Stable descending-confidence order.
    order = np.lexsort((np.arange(len(conf)), -conf))
    y_s = y_true[order]
    p_s = pred[order]

    errors = (p_s != y_s).astype(int)
    k = np.arange(1, len(y_true) + 1)
    risk = np.cumsum(errors) / k
    coverage = k / len(y_true)

    aurc = float(np.mean(risk))

    true_stress = (y_s == 1)
    pred_stress = (p_s == 1)
    total_true_stress = int((y_true == 1).sum())
    tp_retained = np.cumsum(
        (true_stress & pred_stress).astype(int)
    )
    stress_recall = tp_retained / total_true_stress

    fixed = {}
    for target in FIXED_COVERAGES:
        kk = max(
            1,
            min(
                len(y_true),
                int(round(target * len(y_true)))
            )
        )
        fixed[target] = {
            "actual_coverage": float(coverage[kk - 1]),
            "risk": float(risk[kk - 1]),
            "stress_recall": float(stress_recall[kk - 1]),
            "confidence_cutoff": float(conf[order][kk - 1]),
        }

    return aurc, fixed

def metric_bundle(y_true, probs):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true, pred, labels=[0, 1]
    )

    aurc, fixed = selective_metrics(
        y_true, probs
    )

    out = {
        "n": int(len(y_true)),
        "stress_rate": float(np.mean(y_true)),
        "accuracy": float(
            accuracy_score(y_true, pred)
        ),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "brier": float(
            np.mean(
                (probs[:, 1] - y_true) ** 2
            )
        ),
        "nll": float(
            log_loss(
                y_true,
                probs,
                labels=[0, 1]
            )
        ),
        "ece": ece_equal_width(
            y_true, probs
        ),
        "ace": ace_equal_frequency(
            y_true, probs
        ),
        "hce_0.90": int(
            (
                (pred != y_true)
                & (conf >= 0.90)
            ).sum()
        ),
        "coverage_0.90": float(
            (conf >= 0.90).mean()
        ),
        "selective_error_0.90": (
            float(
                (
                    pred[conf >= 0.90]
                    != y_true[conf >= 0.90]
                ).mean()
            )
            if (conf >= 0.90).any()
            else np.nan
        ),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "aurc": aurc,
    }

    for cov, d in fixed.items():
        pct = int(round(cov * 100))
        out[
            f"risk_at_{pct}pct_coverage"
        ] = d["risk"]
        out[
            f"stress_recall_at_{pct}pct_coverage"
        ] = d["stress_recall"]
        out[
            f"confidence_cutoff_at_{pct}pct_coverage"
        ] = d["confidence_cutoff"]

    return out


## 12. Build target-OOD and source-reference predictions

For each held-out domain, this creates three target conditions and three source-reference conditions.


In [ ]:
prediction_parts = []

for heldout in DOMAINS:
    source_train_df, source_cal_df, target_test_df, source_test_df = (
        build_fold_frames(heldout)
    )

    # Vanilla + TS share the same neural logits.
    vanilla = runs[(heldout, "none")]
    T = temperatures[heldout]

    for eval_group, frame, logits, y in [
        (
            "target_OOD",
            target_test_df,
            vanilla["target_logits"],
            vanilla["target_y"],
        ),
        (
            "source_reference",
            source_test_df,
            vanilla["source_logits"],
            vanilla["source_y"],
        ),
    ]:
        for condition, transformed in [
            ("Vanilla", logits),
            ("Vanilla+TS", logits / T),
        ]:
            probs = softmax_np(transformed)
            prediction_parts.append(
                pd.DataFrame({
                    "heldout_domain": heldout,
                    "evaluation_group": eval_group,
                    "condition": condition,
                    "id": frame["id"].to_numpy(),
                    "post_id": frame["post_id"].to_numpy(),
                    "subreddit": frame["subreddit"].to_numpy(),
                    "domain": frame["domain"].to_numpy(),
                    "true_label": y,
                    "p_not_stress": probs[:, 0],
                    "p_stress": probs[:, 1],
                    "predicted_label": probs.argmax(axis=1),
                    "confidence": probs.max(axis=1),
                })
            )

    ls = runs[(heldout, "ls")]

    for eval_group, frame, logits, y in [
        (
            "target_OOD",
            target_test_df,
            ls["target_logits"],
            ls["target_y"],
        ),
        (
            "source_reference",
            source_test_df,
            ls["source_logits"],
            ls["source_y"],
        ),
    ]:
        probs = softmax_np(logits)

        prediction_parts.append(
            pd.DataFrame({
                "heldout_domain": heldout,
                "evaluation_group": eval_group,
                "condition": "LS0.10",
                "id": frame["id"].to_numpy(),
                "post_id": frame["post_id"].to_numpy(),
                "subreddit": frame["subreddit"].to_numpy(),
                "domain": frame["domain"].to_numpy(),
                "true_label": y,
                "p_not_stress": probs[:, 0],
                "p_stress": probs[:, 1],
                "predicted_label": probs.argmax(axis=1),
                "confidence": probs.max(axis=1),
            })
        )

e4_predictions = pd.concat(
    prediction_parts,
    ignore_index=True
)

e4_predictions.to_csv(
    OUT_DIR / "e4_prediction_level_long.csv",
    index=False
)

display(
    e4_predictions.groupby(
        [
            "heldout_domain",
            "evaluation_group",
            "condition",
        ]
    ).size().rename("n").reset_index()
)


## 13. Per-fold domain-shift metrics and target-minus-source deltas

In [ ]:
metric_rows = []

for (
    heldout,
    eval_group,
    condition
), x in e4_predictions.groupby(
    [
        "heldout_domain",
        "evaluation_group",
        "condition",
    ]
):
    probs = x[
        ["p_not_stress", "p_stress"]
    ].to_numpy(float)

    y = x[
        "true_label"
    ].to_numpy(int)

    metric_rows.append({
        "heldout_domain": heldout,
        "evaluation_group": eval_group,
        "condition": condition,
        **metric_bundle(y, probs),
    })

fold_metrics = pd.DataFrame(
    metric_rows
).sort_values(
    [
        "heldout_domain",
        "condition",
        "evaluation_group",
    ]
)

fold_metrics.to_csv(
    OUT_DIR / "e4_lodo_metrics_long.csv",
    index=False
)

delta_rows = []

delta_metrics = [
    "accuracy",
    "f1",
    "brier",
    "nll",
    "ece",
    "ace",
    "hce_0.90",
    "coverage_0.90",
    "selective_error_0.90",
    "aurc",
    "risk_at_90pct_coverage",
    "risk_at_80pct_coverage",
    "risk_at_70pct_coverage",
    "risk_at_50pct_coverage",
]

for heldout in DOMAINS:
    for condition in [
        "Vanilla",
        "Vanilla+TS",
        "LS0.10",
    ]:
        target = fold_metrics[
            (fold_metrics["heldout_domain"] == heldout)
            & (fold_metrics["condition"] == condition)
            & (fold_metrics["evaluation_group"] == "target_OOD")
        ].iloc[0]

        source = fold_metrics[
            (fold_metrics["heldout_domain"] == heldout)
            & (fold_metrics["condition"] == condition)
            & (fold_metrics["evaluation_group"] == "source_reference")
        ].iloc[0]

        row = {
            "heldout_domain": heldout,
            "condition": condition,
        }

        for metric in delta_metrics:
            row[f"delta_target_minus_source_{metric}"] = (
                target[metric] - source[metric]
            )

        delta_rows.append(row)

shift_deltas = pd.DataFrame(delta_rows)

shift_deltas.to_csv(
    OUT_DIR / "e4_lodo_shift_deltas.csv",
    index=False
)

display(fold_metrics[[
    "heldout_domain",
    "evaluation_group",
    "condition",
    "n",
    "f1",
    "brier",
    "nll",
    "ece",
    "aurc",
]])

print("\nTarget minus source shift deltas:")
display(shift_deltas)


## 14. Macro-domain summary and pooled LODO OOD evaluation

### Macro-domain summary
Each of the five target domains receives equal weight.

### Pooled LODO OOD
Every official-test example appears exactly once as a target example:
- an abuse example is predicted by the model that never saw abuse,
- an anxiety example by the model that never saw anxiety,
- and so forth.

This yields one pooled OOD prediction set of **715 examples per condition**.


In [ ]:
target_metrics = fold_metrics[
    fold_metrics["evaluation_group"] == "target_OOD"
].copy()

summary_metrics = [
    "accuracy",
    "f1",
    "brier",
    "nll",
    "ece",
    "ace",
    "hce_0.90",
    "coverage_0.90",
    "selective_error_0.90",
    "aurc",
    "risk_at_90pct_coverage",
    "risk_at_80pct_coverage",
    "risk_at_70pct_coverage",
    "risk_at_50pct_coverage",
]

macro_rows = []

for condition, x in target_metrics.groupby("condition"):
    row = {
        "condition": condition,
        "domains": int(x["heldout_domain"].nunique()),
    }

    for m in summary_metrics:
        row[f"macro_mean_{m}"] = float(
            x[m].mean()
        )
        row[f"macro_std_{m}"] = float(
            x[m].std(ddof=1)
        )

    # Worst-domain is descriptive, not a method ranking.
    row["max_domain_brier"] = float(x["brier"].max())
    row["max_domain_nll"] = float(x["nll"].max())
    row["max_domain_ece"] = float(x["ece"].max())
    row["max_domain_aurc"] = float(x["aurc"].max())
    row["min_domain_f1"] = float(x["f1"].min())

    macro_rows.append(row)

macro_summary = pd.DataFrame(macro_rows)
macro_summary.to_csv(
    OUT_DIR / "e4_macro_domain_summary.csv",
    index=False
)

pooled_target = e4_predictions[
    e4_predictions["evaluation_group"] == "target_OOD"
].copy()

# Each official-test ID must appear once per condition in target-OOD pool.
target_counts = pooled_target.groupby(
    "condition"
)["id"].nunique()

if not target_counts.eq(EXPECTED_TEST_N).all():
    raise RuntimeError(
        f"Pooled target OOD does not contain exactly {EXPECTED_TEST_N} "
        f"unique test IDs per condition:\n{target_counts}"
    )

pooled_rows = []

for condition, x in pooled_target.groupby("condition"):
    probs = x[
        ["p_not_stress", "p_stress"]
    ].to_numpy(float)

    y = x[
        "true_label"
    ].to_numpy(int)

    pooled_rows.append({
        "condition": condition,
        **metric_bundle(y, probs),
    })

pooled_metrics = pd.DataFrame(
    pooled_rows
)

pooled_metrics.to_csv(
    OUT_DIR / "e4_pooled_lodo_ood_metrics.csv",
    index=False
)

pooled_target.to_csv(
    OUT_DIR / "e4_pooled_lodo_target_predictions_for_E7.csv",
    index=False
)

display(macro_summary)
print("\nPooled LODO OOD metrics:")
display(pooled_metrics)


## 15. Temperature Scaling rank-invariance audit under domain shift

A positive scalar temperature should preserve:
- class predictions,
- confidence ordering up to ties,
- rank-based AURC.

The cell records AURC equality per held-out target domain.


In [ ]:
invariance_rows = []

for heldout in DOMAINS:
    for eval_group in [
        "target_OOD",
        "source_reference",
    ]:
        a = e4_predictions[
            (e4_predictions["heldout_domain"] == heldout)
            & (e4_predictions["evaluation_group"] == eval_group)
            & (e4_predictions["condition"] == "Vanilla")
        ].copy()

        b = e4_predictions[
            (e4_predictions["heldout_domain"] == heldout)
            & (e4_predictions["evaluation_group"] == eval_group)
            & (e4_predictions["condition"] == "Vanilla+TS")
        ].copy()

        a = a.sort_values("id").reset_index(drop=True)
        b = b.sort_values("id").reset_index(drop=True)

        same_predictions = bool(
            np.array_equal(
                a["predicted_label"].to_numpy(),
                b["predicted_label"].to_numpy(),
            )
        )

        ma = metric_bundle(
            a["true_label"].to_numpy(int),
            a[["p_not_stress", "p_stress"]].to_numpy(float),
        )
        mb = metric_bundle(
            b["true_label"].to_numpy(int),
            b[["p_not_stress", "p_stress"]].to_numpy(float),
        )

        invariance_rows.append({
            "heldout_domain": heldout,
            "evaluation_group": eval_group,
            "same_predicted_labels": same_predictions,
            "aurc_vanilla": ma["aurc"],
            "aurc_temperature": mb["aurc"],
            "absolute_aurc_difference": abs(
                ma["aurc"] - mb["aurc"]
            ),
        })

invariance = pd.DataFrame(
    invariance_rows
)

invariance.to_csv(
    OUT_DIR / "e4_temperature_aurc_invariance.csv",
    index=False
)

display(invariance)


## 16. Publication table

In [ ]:
table_e4 = target_metrics[[
    "heldout_domain",
    "condition",
    "n",
    "stress_rate",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "brier",
    "nll",
    "ece",
    "ace",
    "hce_0.90",
    "coverage_0.90",
    "selective_error_0.90",
    "aurc",
    "risk_at_90pct_coverage",
    "risk_at_80pct_coverage",
    "risk_at_70pct_coverage",
    "risk_at_50pct_coverage",
]].copy()

table_e4.to_csv(
    OUT_DIR / "Table_E4_LODO_Domain_Shift.csv",
    index=False
)

display(table_e4)


## 17. Figures

In [ ]:
import matplotlib.pyplot as plt

for metric, ylabel, filename in [
    ("brier", "Brier score", "Figure_E4_Brier_by_Heldout_Domain"),
    ("nll", "Negative log-likelihood", "Figure_E4_NLL_by_Heldout_Domain"),
    ("ece", "ECE", "Figure_E4_ECE_by_Heldout_Domain"),
    ("aurc", "AURC", "Figure_E4_AURC_by_Heldout_Domain"),
]:
    fig, ax = plt.subplots(figsize=(8.5, 5.5))

    for condition in [
        "Vanilla",
        "Vanilla+TS",
        "LS0.10",
    ]:
        x = target_metrics[
            target_metrics["condition"] == condition
        ]

        x = (
            x.set_index("heldout_domain")
            .loc[DOMAINS]
            .reset_index()
        )

        ax.plot(
            x["heldout_domain"],
            x[metric],
            marker="o",
            label=condition,
        )

    ax.set_xlabel("Held-out target domain")
    ax.set_ylabel(ylabel)
    ax.set_title(
        f"LODO Domain Shift — {ylabel}"
    )
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()

    fig.savefig(
        OUT_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        OUT_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )
    plt.show()


## 18. Evidence hashes and final E4 archive

Run this only after every preceding E4 cell succeeds.


In [ ]:
def hash_output_files(directory):
    rows = []

    for p in sorted(Path(directory).iterdir()):
        if not p.is_file():
            continue
        if p.name in {
            "E4_OUTPUT_HASHES.csv",
            "E4_MANIFEST.json",
        }:
            continue

        rows.append({
            "filename": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

    return pd.DataFrame(rows)

hashes = hash_output_files(OUT_DIR)

hashes.to_csv(
    OUT_DIR / "E4_OUTPUT_HASHES.csv",
    index=False
)

manifest = {
    "package": "CalibBERT_v2_E4_LODO_Domain_Shift",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_train_sha256": EXPECTED_TRAIN_SHA256,
    "canonical_test_sha256": EXPECTED_TEST_SHA256,
    "heldout_domains": DOMAINS,
    "conditions": [
        "Vanilla",
        "Vanilla+TS",
        "LS0.10",
    ],
    "official_test_target_pool_n_per_condition": EXPECTED_TEST_N,
    "files": hashes.to_dict(orient="records"),
}

(OUT_DIR / "E4_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2)
)

ZIP_PATH = PROJECT_ROOT / "E4_RESULTS.zip"

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

print("Final E4 archive:", ZIP_PATH)
print("Output files:", len(list(OUT_DIR.iterdir())))


## 19. E4 completion gate

E4 is ready for audit only when all items below are satisfied:

- [ ] canonical raw-data hashes pass
- [ ] original 10-subreddit → 5-domain taxonomy is used unchanged
- [ ] archived post-grouped fit/validation/calibration assignment is reused
- [ ] E4 design lock is written before LODO training
- [ ] BERT revision is locked to an immutable Hugging Face commit
- [ ] all 5 held-out domains complete
- [ ] each held-out domain is excluded from both classifier training and TS calibration
- [ ] Vanilla and LS0.10 are paired with seed 42
- [ ] all logits are finite
- [ ] source-domain reference and target-OOD metrics are both exported
- [ ] target-minus-source shift deltas are exported
- [ ] pooled LODO target predictions contain exactly 715 unique official-test instances per condition
- [ ] temporal metadata are documented without claiming isolated temporal robustness
- [ ] no method/backbone is selected using target official-test performance
- [ ] `E4_RESULTS.zip` is downloaded or safely retained in Google Drive

### Files to send for final E4 audit
1. `E4_RESULTS.zip`
2. `Table_E4_LODO_Domain_Shift.csv`
3. `e4_pooled_lodo_ood_metrics.csv`
4. `e4_lodo_shift_deltas.csv`
5. `e4_temperature_scaling_parameters.csv`
6. `E4_ANALYSIS_LOCK.json`

### After E4
If E4 passes, the next stage is **E6 semantic/topic stability**.  
Formal post-level cluster bootstrap and multiplicity-controlled paired inference remain reserved for **E7**.
